# FASHN VTON v1.5 — GPU evaluation notebook

This notebook is for **permitted non-commercial research/evaluation only**. The VTON repository is Apache-2.0, but its inference pipeline also uses FASHN Human Parser, whose license points to the NVIDIA SegFormer license and restricts use to non-commercial research/evaluation. This is **not cleared for the boutique's commercial production use**. Review all current component and model-weight licenses before proceeding; setting the acknowledgement below does not grant rights. If your planned use is commercial, stop here and select a commercially authorized model.

Upstream references: [FASHN VTON v1.5](https://github.com/fashn-AI/fashn-vton-1.5), [model card](https://huggingface.co/fashn-ai/fashn-vton-1.5), [parser license notice](https://github.com/fashn-AI/fashn-human-parser#license), [NVIDIA SegFormer license](https://github.com/NVlabs/SegFormer/blob/master/LICENSE).

Use only images you have permission to process. Kaggle sessions are transient and are not secure production environments. Do not publish customer images, credentials, or private notebook links.

## Before running

1. In Kaggle, open **Settings → Accelerator → GPU** and enable a GPU runtime.
2. Add a private Kaggle dataset containing consented person and garment images, or use images you are otherwise authorized to use. Do not use a public dataset for private/customer photos.
3. Read the license warning above. The next code cell intentionally stops unless you change the explicit opt-in flag after confirming the use is allowed.
4. Installation and weight download take several minutes and require internet access; model and parser weights use several GB of storage/cache combined.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print('Python:', platform.python_version())
print('GPU utility:', shutil.which('nvidia-smi'))
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch is not installed yet; the official project setup installs its dependencies.')

In [ ]:
# Change to True only after reviewing all licenses and confirming this is permitted
# non-commercial research/evaluation. This is an acknowledgement, not a license grant.
NON_COMMERCIAL_RESEARCH_ACKNOWLEDGED = False
if not NON_COMMERCIAL_RESEARCH_ACKNOWLEDGED:
    raise RuntimeError('Review the license notice and explicitly acknowledge permitted non-commercial research first.')

## Install the pinned upstream code and weights

The source revision below was resolved from the official repository. The upstream install/download commands include third-party components; execute them only after the license review and opt-in above. Weight download includes the human-parser weights, which may be cached automatically by the upstream library.

In [ ]:
REPOSITORY = 'https://github.com/fashn-AI/fashn-vton-1.5.git'
REVISION = '7c0f10af3f91ad4048fe9729c470a13ef905d25a'
WORK_DIR = Path('/kaggle/working/fashn-vton-1.5')
WEIGHTS_DIR = WORK_DIR / 'weights'
HF_HOME = Path('/kaggle/working/huggingface-cache')
os.environ['HF_HOME'] = str(HF_HOME)

if not WORK_DIR.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(WORK_DIR)], check=True)
subprocess.run(['git', '-C', str(WORK_DIR), 'fetch', '--depth', '1', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(WORK_DIR), 'checkout', '--detach', REVISION], check=True)
print('Pinned source revision:', subprocess.check_output(
    ['git', '-C', str(WORK_DIR), 'rev-parse', 'HEAD'], text=True
).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(WORK_DIR)], check=True)
if not (WORK_DIR / 'scripts' / 'download_weights.py').is_file():
    raise FileNotFoundError('The pinned repository does not contain its official weight-download script.')
subprocess.run([
    sys.executable, str(WORK_DIR / 'scripts' / 'download_weights.py'),
    '--weights-dir', str(WEIGHTS_DIR)
], check=True)
print('Weights directory:', WEIGHTS_DIR)

## Select authorized sample images

In Kaggle, add a dataset through the notebook's **Add Data** panel. Set the two paths below to files in that dataset. The person image should clearly show the wearer; the garment image may be a worn-model photo or a flat-lay product shot. The input type and category must match the image. Supported categories are only `tops`, `bottoms`, and `one-pieces`; do not assume saree, lehenga, dupatta, or other Indian garment styling is supported or reliable.

In [ ]:
PERSON_IMAGE = Path('/kaggle/input/your-private-dataset/person.jpg')
GARMENT_IMAGE = Path('/kaggle/input/your-private-dataset/garment.jpg')
CATEGORY = 'tops'  # tops | bottoms | one-pieces
GARMENT_PHOTO_TYPE = 'flat-lay'  # model | flat-lay
NUM_TIMESTEPS = 30
GUIDANCE_SCALE = 1.5
SEED = 42

if not PERSON_IMAGE.is_file() or not GARMENT_IMAGE.is_file():
    raise FileNotFoundError('Update PERSON_IMAGE and GARMENT_IMAGE to consented files in your attached Kaggle dataset.')
if CATEGORY not in {'tops', 'bottoms', 'one-pieces'}:
    raise ValueError('CATEGORY must be tops, bottoms, or one-pieces.')
if GARMENT_PHOTO_TYPE not in {'model', 'flat-lay'}:
    raise ValueError('GARMENT_PHOTO_TYPE must be model or flat-lay.')
print('Person image:', PERSON_IMAGE.name, PERSON_IMAGE.stat().st_size, 'bytes')
print('Garment image:', GARMENT_IMAGE.name, GARMENT_IMAGE.stat().st_size, 'bytes')

## Run and time inference

The model produces a 576x864 RGB image. GPU inference time varies by GPU and input. This cell records elapsed time and peak allocated CUDA memory when available. It catches out-of-memory errors to report a useful message, then releases cached CUDA allocations; it does not silently substitute a fake result.

In [ ]:
import gc
import time
import torch
from PIL import Image
from fashn_vton import TryOnPipeline

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required for this evaluation notebook. Enable a Kaggle GPU accelerator.')
pipeline = TryOnPipeline(weights_dir=str(WEIGHTS_DIR))
with Image.open(PERSON_IMAGE) as image:
    person = image.convert('RGB')
with Image.open(GARMENT_IMAGE) as image:
    garment = image.convert('RGB')

try:
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    result = pipeline(
        person_image=person,
        garment_image=garment,
        category=CATEGORY,
        garment_photo_type=GARMENT_PHOTO_TYPE,
        num_timesteps=NUM_TIMESTEPS,
        guidance_scale=GUIDANCE_SCALE,
        seed=SEED,
    )
    elapsed = time.perf_counter() - started
    if not result.images:
        raise RuntimeError('The model returned no output images.')
    output_image = result.images[0].convert('RGB')
    print(f'Inference duration: {elapsed:.2f} seconds')
    print(f'CUDA peak allocated: {torch.cuda.max_memory_allocated() / (1024 ** 2):.1f} MiB')
    print('Output dimensions:', output_image.size)
except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
    is_oom = isinstance(error, torch.cuda.OutOfMemoryError) or 'out of memory' in str(error).lower()
    if is_oom:
        gc.collect()
        torch.cuda.empty_cache()
        print('GPU out of memory. Restart the notebook session, check accelerator memory, and retry with a suitable GPU.')
    raise
finally:
    person.close()
    garment.close()

In [ ]:
from IPython.display import display

display(output_image)
OUTPUT_PATH = Path('/kaggle/working/fashn-vton-result.png')
output_image.save(OUTPUT_PATH, format='PNG')
print('Saved result to:', OUTPUT_PATH)

## Cleanup and privacy

The result is stored in this session's `/kaggle/working` area. Save only results you are authorized to retain and avoid downloading/sharing them outside the permitted evaluation. At the end, close the images and release GPU memory; ending the Kaggle session removes its ephemeral working storage. Do not delete the attached input dataset from code.

In [ ]:
if 'output_image' in globals():
    output_image.close()
for name in ('pipeline', 'result', 'person', 'garment'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Image objects and cached GPU allocations released. End the session to remove temporary working files.')

## Google Colab alternative

Choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then follow the same license gate, pinned repository install, weight download, and inference cells. Replace `/kaggle/working` paths with `/content` and provide authorized image paths from Colab's Files panel or a private Drive mount. Do not publish a notebook containing personal images or access tokens. Colab is also an evaluation environment, not a production service.